# 06 - Artefak penulisan: gambar dan data tabel

Dijalankan SETELAH `05_final_benchmark.ipynb`. Dua keluaran, seluruhnya dibangkitkan langsung dari berkas log running tanpa angka yang diketik manual:

1. **Gambar 4.1-4.8** ke `outputs/tuning/artifacts/gambar/` sebagai PNG 300 dpi dan PDF vektor.
2. **Data mentah Tabel 4.1-4.19 dan L.1** ke `outputs/tuning/artifacts/data_tabel.xlsx`: satu sheet per nomor tabel dengan nilai presisi penuh, ditambah sheet `keterangan` (judul, sumber berkas, catatan, status). Tabel siap tempel TIDAK dibuat; tabel disusun manual saat penulisan dari data ini.

Artefak yang berkas lognya belum ada (misalnya sebelum 05 dijalankan) dilewati dengan keterangan, jadi notebook ini aman dijalankan ulang.

In [ ]:
from pathlib import Path

# Folder keluaran kampanye 03a-05
OUT_DIR = Path("../outputs/tuning")
# metadata.json split dan statistik preprocessing dari 02
METADATA_PATH = Path("../data/processed/metadata.json")
STATS_PATH = Path("../outputs/preprocessing/preprocessing_stats.json")
FIGURE_DIR = OUT_DIR / "artifacts" / "gambar"
TABLE_PATH = OUT_DIR / "artifacts" / "data_tabel.xlsx"

TIE_THRESHOLD_PP = 0.15
METRIC_PRECISION = 6
# Kriteria sukses, sama dengan 05
MAX_F1_GAP_PP = 3.0
MIN_PARAM_REDUCTION_PCT = 90.0
MIN_TIME_REDUCTION_PCT = 50.0
MIN_CRITERIA = 2
# Jumlah konfigurasi teratas di Tabel 4.8, 4.10, dan 4.11
TOP_N = 3

SCENARIO_LABELS = {"rma": "RM-a", "rmb": "RM-b", "rmc": "RM-c"}
CLASS_NAMES = ["non-judi (0)", "judi (1)"]
# batch_id tahap grid yang dibaca gambar dan tabel
RMA_GRID_BATCH = "rma_tahap1_grid"
RMB_CAPACITY_BATCHES = ["rmb_tuning_grid", "rmb_tuning_grid_stage1b"]
RMA_STAGES = {
    "": "Tahap 1: grid lr x epoch x batch",
    "rma_tahap1_grid": "Tahap 1: grid lr x epoch x batch",
    "rma_tahap2_coordinate": "Tahap 2: coordinate descent",
}
RMB_STAGES = {
    "rmb_tuning_grid": "Tahap 1: arsitektur head",
    "rmb_tuning_grid_stage1b": "Tahap 1B: perluasan dimensi tersembunyi",
    "rmb_tuning_grid_stage2": "Tahap 2: grid lr x epoch",
    "rmb_tuning_grid_stage3": "Tahap 3: coordinate descent",
}
RMC_STAGES = {
    "rmc_grid_seluruh_head": "Tahap 1: seluruh head RM-b x alpha x k",
    "rmc_tahap2_weighting": "Tahap 2: verifikasi skema pembobotan",
}
LATENCY_COMPONENTS = ["encoder", "classification head", "retrieval FAISS", "fusi"]

## 1. Load log kampanye

Pengakses di bawah melempar `FileNotFoundError` bila berkasnya belum ada, sehingga gambar atau tabel yang bergantung padanya dilewati dengan keterangan.

In [ ]:
import hashlib
import json
import time

import numpy as np
import pandas as pd
from IPython.display import Image, display


def load_required_json(path: Path) -> dict:
    """Baca berkas JSON wajib.

    Raises:
        FileNotFoundError: Bila berkas belum ada atau kosong.
    """
    content = json.loads(path.read_text(encoding="utf-8")) if path.exists() else None
    if not content:
        raise FileNotFoundError(f"{path.name} belum ada ({path})")
    return content


def load_runs(scenario: str) -> pd.DataFrame:
    """Riwayat run satu skenario.

    Raises:
        FileNotFoundError: Bila `runs_{scenario}.csv` belum ada.
    """
    path = OUT_DIR / f"runs_{scenario}.csv"
    if not path.exists():
        raise FileNotFoundError(f"runs_{scenario}.csv belum ada atau kosong di {OUT_DIR}")
    return pd.read_csv(path)


def load_best() -> dict:
    """Isi `best.json`."""
    return load_required_json(OUT_DIR / "best.json")


def load_metrics_csv(filename: str) -> pd.DataFrame:
    """Tabel keluaran 05 di `metrics/`.

    Raises:
        FileNotFoundError: Bila 05 belum dijalankan.
    """
    path = OUT_DIR / "metrics" / filename
    if not path.exists():
        raise FileNotFoundError(f"metrics/{filename} belum ada; jalankan 05_final_benchmark.ipynb lebih dulu")
    return pd.read_csv(path)


def load_champion_row(scenario: str) -> pd.Series:
    """Baris riwayat juara satu skenario."""
    runs = load_runs(scenario)
    return runs[runs["run_id"] == int(load_best()[scenario]["run_id"])].iloc[0]

## 2. Ringkasan turunan

Fungsi di bawah sama dengan di 03c dan 05: grid RM-c dengan gain terhadap `alpha=0`, ringkasan RAC per head, peringkat run, dan metrik test dari prediksi per baris. Gambar dan tabel membaca angka yang sama dari sini.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """Accuracy, precision/recall/F1 macro, dan precision/recall/F1 kelas judi."""
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p_class, r_class, f_class, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f_macro),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_class1": float(f_class[1]),
        "precision_class1": float(p_class[1]),
        "recall_class1": float(r_class[1]),
    }


def to_metric_columns(metrics: dict[str, float]) -> dict[str, float]:
    """Kolom metrik validation untuk baris riwayat run."""
    return {
        "val_f1_macro": round(metrics["f1_macro"], METRIC_PRECISION),
        "val_acc": round(metrics["accuracy"], METRIC_PRECISION),
        "val_precision_macro": round(metrics["precision_macro"], METRIC_PRECISION),
        "val_recall_macro": round(metrics["recall_macro"], METRIC_PRECISION),
        "val_f1_judi": round(metrics["f1_class1"], METRIC_PRECISION),
    }

def compute_content_hash(body: dict) -> str:
    """SHA-256 isi kanonik `candidates.json` di luar field hash itu sendiri."""
    canonical = json.dumps({key: value for key, value in body.items() if key != "content_sha256"},
                           sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def to_rmc_grid(runs_rmc: pd.DataFrame) -> pd.DataFrame:
    """Baris grid (`weighting=similarity`) beserta F1 tanpa RAC (`alpha=0`) dan gain per head."""
    grid = runs_rmc[runs_rmc["weighting"] == "similarity"].copy()
    grid["rmb_run_id"] = grid["rmb_run_id"].astype(int)
    baseline = (
        grid[grid["alpha"] == 0.0].sort_values(["rmb_run_id", "k"])
        .groupby("rmb_run_id")[["val_f1_macro", "val_f1_judi"]].first()
        .rename(columns={"val_f1_macro": "f1_no_rac", "val_f1_judi": "f1_judi_no_rac"})
    )
    missing = sorted(set(grid["rmb_run_id"]) - set(baseline.index))
    if missing:
        raise ValueError(f"Head RM-b {missing} tidak punya baris alpha=0 di grid RM-c")
    grid = grid.join(baseline, on="rmb_run_id")
    grid["gain_pp"] = (grid["val_f1_macro"] - grid["f1_no_rac"]) * 100
    grid["gain_judi_pp"] = (grid["val_f1_judi"] - grid["f1_judi_no_rac"]) * 100
    return grid


def find_shared_config(grid: pd.DataFrame) -> tuple[float, int]:
    """(alpha, k) dengan rata-rata gain tertinggi di seluruh head; seri: alpha lalu k lebih kecil."""
    means = grid.groupby(["alpha", "k"])["gain_pp"].mean().reset_index()
    best = means.sort_values(["gain_pp", "alpha", "k"], ascending=[False, True, True]).iloc[0]
    return float(best["alpha"]), int(best["k"])


def summarize_rac_per_head(runs_rmc: pd.DataFrame, runs_rmb: pd.DataFrame) -> pd.DataFrame:
    """Satu baris per head: tanpa RAC, konfigurasi terbaik head itu, dan konfigurasi bersama."""
    grid = to_rmc_grid(runs_rmc)
    alpha_shared, k_shared = find_shared_config(grid)
    best = (grid.sort_values(["val_f1_macro", "val_f1_judi", "k", "alpha"], ascending=[False, False, True, True])
            .groupby("rmb_run_id", sort=False).head(1).set_index("rmb_run_id"))
    at_shared = grid[(grid["alpha"] == alpha_shared) & (grid["k"] == k_shared)].set_index("rmb_run_id")
    summary = pd.DataFrame({
        "f1_no_rac": best["f1_no_rac"], "f1_judi_no_rac": best["f1_judi_no_rac"],
        "best_alpha": best["alpha"], "best_k": best["k"], "f1_best": best["val_f1_macro"],
        "gain_best_pp": best["gain_pp"], "gain_judi_best_pp": best["gain_judi_pp"],
        "f1_shared": at_shared["val_f1_macro"], "gain_shared_pp": at_shared["gain_pp"],
    })
    summary["shared_alpha"] = alpha_shared
    summary["shared_k"] = k_shared
    summary = summary.join(runs_rmb.set_index("run_id")[["head_arch", "hidden_dim", "lr", "epochs", "trainable_params"]])
    return summary.rename_axis("rmb_run_id").reset_index()


def rank_runs(runs: pd.DataFrame, scenario: str) -> pd.DataFrame:
    """Urutkan run: F1-macro, F1 judi, lalu training time (RM-c: biaya head, k, alpha) lebih kecil."""
    keys = ["val_f1_macro", "val_f1_judi"] + (["head_train_time_s", "k", "alpha"] if scenario == "rmc" else ["train_time_s"])
    return runs.sort_values(keys, ascending=[False, False] + [True] * (len(keys) - 2), kind="mergesort").reset_index(drop=True)


def rank_rmc_runs(runs: pd.DataFrame) -> pd.DataFrame:
    """Urutkan run RM-c: F1-macro, F1 judi, head lebih murah, k dan alpha lebih kecil."""
    return runs.sort_values(["val_f1_macro", "val_f1_judi", "head_trainable_params", "k", "alpha"],
                            ascending=[False, False, True, True, True], kind="mergesort").reset_index(drop=True)


def load_candidates(path: Path) -> dict:
    """Baca `candidates.json` dan pastikan hash isinya cocok.

    Raises:
        FileNotFoundError: Bila kandidat belum ditetapkan.
        ValueError: Bila hash isi tidak cocok.
    """
    body = load_required_json(path)
    if compute_content_hash(body) != body.get("content_sha256"):
        raise ValueError(f"Hash {path} tidak cocok: isi kandidat berubah setelah ditetapkan")
    return body


def compute_test_results() -> dict[str, dict[str, float]]:
    """Metrik test dan confusion matrix (judi = positif) tiap skenario dari `final_predictions.csv`."""
    predictions = load_metrics_csv("final_predictions.csv")
    results = {}
    for scenario in SCENARIO_LABELS:
        truth, pred = predictions["label"].to_numpy(), predictions[f"pred_{scenario}"].to_numpy()
        tn, fp, fn, tp = confusion_matrix(truth, pred, labels=[0, 1]).ravel()
        results[scenario] = {**compute_metrics(truth, pred), "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)}
    return results


def to_head_label(head_arch: str, hidden_dim: object) -> str:
    """`linear` atau `MLP <hidden_dim>`."""
    return "linear" if head_arch == "linear" else f"MLP {int(hidden_dim)}"


def to_head_group(head_arch: str, hidden_dim: int) -> str:
    """Kelompok head menurut arsitektur, ditetapkan sebelum melihat hasil."""
    if head_arch == "linear":
        return "Linear"
    if int(hidden_dim) < 1024:
        return "MLP hidden dim < 1024"
    return f"MLP hidden dim = {int(hidden_dim)}"

## 3. Gaya gambar

Aturan gambar skripsi: Liberation Serif, lebar 5,5 inci, tanpa judul di dalam gambar, satu keluarga warna biru-nila yang tetap terbaca bila dicetak hitam putih, jingga hanya untuk penanda konfigurasi final, dan koma sebagai pemisah desimal.

Pembulatan bawaan Python (dan f-string) memakai representasi biner float, sehingga 0,125 bisa menjadi 0,12. Anotasi gambar memakai `Decimal` dengan `ROUND_HALF_UP` supaya angka di gambar sama dengan pembulatan manual di tabel.

In [ ]:
from decimal import ROUND_HALF_UP, Decimal

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter

FIGURE_WIDTH_IN = 5.5
FIGURE_DPI = 300
# Gradasi biru-nila terang ke gelap; luminansinya menurun monoton sehingga tetap terbedakan hitam putih
BLUE_RAMP = ["#dde5f4", "#b2c3e5", "#7f99d0", "#5270b8", "#324c96", "#1c2b63"]
FINAL_ORANGE = "#e07b1a"
WARM_GRAY = "#8a7d70"
NEUTRAL = "#f4f3f1"
INK = "#222222"
MUTED_INK = "#666666"
LABEL_BOX = {"boxstyle": "square,pad=0.1", "facecolor": "white", "edgecolor": "none"}
SCENARIO_COLORS = {"RM-a": BLUE_RAMP[5], "RM-b": BLUE_RAMP[3], "RM-c": BLUE_RAMP[1]}
SCENARIO_MARKERS = {"RM-a": "o", "RM-b": "s", "RM-c": "D"}
STYLE = {
    "font.family": "serif", "font.serif": ["Liberation Serif", "Times New Roman", "DejaVu Serif"],
    "font.size": 8.5, "axes.labelsize": 8.5, "axes.titlesize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 7.5, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED_INK, "axes.labelcolor": INK, "xtick.color": MUTED_INK, "ytick.color": MUTED_INK,
    "axes.grid": False, "grid.color": "#e3e3e3", "grid.linewidth": 0.5, "legend.frameon": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "axes.formatter.use_locale": False,
    "mathtext.fontset": "custom", "mathtext.rm": "Liberation Serif",
    "mathtext.it": "Liberation Serif:italic", "mathtext.bf": "Liberation Serif:bold",
}


def round_half_up(value: float, decimals: int) -> Decimal:
    """Bulatkan setengah ke atas berdasarkan representasi desimal terpendek float."""
    return Decimal(repr(float(value))).quantize(Decimal(1).scaleb(-decimals), rounding=ROUND_HALF_UP)


def format_number(value: float, decimals: int = 2, sign: bool = False) -> str:
    """Format gaya Indonesia, contoh: 1234.567 -> "1.234,57"; `sign=True` memberi `+` pada nilai positif."""
    rounded = round_half_up(value, decimals)
    if rounded == 0:
        rounded = abs(rounded)
    text = f"{rounded:,.{decimals}f}"
    if sign and rounded > 0:
        text = "+" + text
    return text.replace(",", "_").replace(".", ",").replace("_", ".")


def format_scientific(value: float) -> str:
    """Learning rate sebagai mathtext, contoh: 2e-05 -> "$2 \\times 10^{-5}$"."""
    exponent = int(np.floor(np.log10(abs(value))))
    mantissa = value / 10**exponent
    digits = format_number(mantissa, 0 if np.isclose(mantissa, round(mantissa)) else 1)
    return rf"${digits.replace(',', '{,}')} \times 10^{{{exponent}}}$"


def number_formatter(decimals: int) -> FuncFormatter:
    """Formatter sumbu matplotlib dengan koma desimal."""
    return FuncFormatter(lambda value, _: format_number(value, decimals))


def percent_log_formatter() -> FuncFormatter:
    """Formatter sumbu persen logaritmik, contoh: "0,001%", "1%", "100%"."""
    def label(value: float, _position: int) -> str:
        decimals = max(0, -int(np.floor(np.log10(value)))) if value > 0 else 0
        return f"{format_number(value, decimals)}%"
    return FuncFormatter(label)


def pick_text_color(rgba: tuple) -> str:
    """Tinta putih di atas warna gelap, tinta gelap di atas warna terang."""
    red, green, blue = rgba[:3]
    return "white" if 0.2126 * red + 0.7152 * green + 0.0722 * blue < 0.5 else INK


def annotate_cells(ax, image, values: np.ndarray, formatter) -> None:
    """Tulis nilai tiap sel heatmap dengan tinta yang kontras terhadap warnanya."""
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            if not np.isnan(values[i, j]):
                ax.text(j, i, formatter(values[i, j]), ha="center", va="center", fontsize=7,
                        color=pick_text_color(image.cmap(image.norm(values[i, j]))))


def frame_value(ax, pivot: pd.DataFrame, row_value: float, column_value: float) -> None:
    """Beri bingkai jingga pada sel pivot yang cocok dengan konfigurasi final."""
    rows = np.flatnonzero(np.isclose(pivot.index.astype(float), float(row_value)))
    columns = np.flatnonzero(np.isclose(pivot.columns.astype(float), float(column_value)))
    if len(rows) and len(columns):
        ax.add_patch(Rectangle((columns[0] - 0.5, rows[0] - 0.5), 1, 1, fill=False,
                               edgecolor=FINAL_ORANGE, linewidth=1.8, zorder=5))


def clear_spines(ax) -> None:
    """Sembunyikan garis tepi dan tick sumbu heatmap."""
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(length=0)


def save_figure(fig: plt.Figure, stem: str) -> list[Path]:
    """Simpan gambar sebagai PNG 300 dpi dan PDF vektor."""
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    paths = [FIGURE_DIR / f"{stem}.png", FIGURE_DIR / f"{stem}.pdf"]
    for path in paths:
        fig.savefig(path, dpi=FIGURE_DPI)
    plt.close(fig)
    return paths

## 4. Gambar 4.1-4.4: hasil tuning pada validation

- **4.1** Heatmap F1-macro validation grid RM-a, satu panel per batch; bingkai jingga = konfigurasi final.
- **4.2** Kurva kapasitas head RM-b: F1-macro validation terhadap trainable params, dengan kenaikan marjinal per ruas.
- **4.3** Heatmap divergen selisih RAC terhadap tanpa RAC pada head juara RM-c; arsiran = di dalam ambang seri.
- **4.4** Dumbbell kontribusi RAC per head RM-b: tanpa RAC, konfigurasi bersama, dan konfigurasi terbaik per head.

In [ ]:
def plot_figure_4_1() -> list[Path]:
    """Heatmap F1-macro validation grid RM-a, satu panel per batch."""
    runs = load_runs("rma")
    grid = runs[runs["batch_id"] == RMA_GRID_BATCH]
    if grid.empty:
        raise FileNotFoundError(f"runs_rma.csv belum memuat batch {RMA_GRID_BATCH!r}")
    final = load_best()["rma"]["config"]
    batches = sorted(grid["batch"].unique())
    low, high = grid["val_f1_macro"].min(), grid["val_f1_macro"].max()
    cmap = LinearSegmentedColormap.from_list("biru_nila", BLUE_RAMP)
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(1, len(batches), figsize=(FIGURE_WIDTH_IN, 2.6), squeeze=False, constrained_layout=True)
        for ax, batch in zip(axes[0], batches, strict=True):
            pivot = grid[grid["batch"] == batch].pivot_table(index="lr", columns="epochs", values="val_f1_macro", aggfunc="max")
            image = ax.imshow(pivot.values, cmap=cmap, vmin=low, vmax=high, aspect="auto")
            annotate_cells(ax, image, pivot.values, lambda v: format_number(v, 4))
            ax.set_xticks(range(len(pivot.columns)), [str(int(c)) for c in pivot.columns])
            ax.set_yticks(range(len(pivot.index)), [format_scientific(r) for r in pivot.index])
            ax.set_xlabel("epoch")
            ax.set_ylabel("learning rate" if ax is axes[0][0] else "")
            ax.set_title(f"batch {int(batch)}")
            clear_spines(ax)
            if int(final["batch"]) == int(batch):
                frame_value(ax, pivot, final["lr"], final["epochs"])
        colorbar = fig.colorbar(image, ax=axes[0].tolist(), fraction=0.05, pad=0.02)
        colorbar.set_label(f"F1-macro validation (rentang {format_number((high - low) * 100, 2)} pp)")
        colorbar.ax.yaxis.set_major_formatter(number_formatter(3))
        return save_figure(fig, "gambar_4_1")


def plot_figure_4_2() -> list[Path]:
    """Kurva kapasitas head RM-b: F1-macro validation terhadap trainable params."""
    runs = load_runs("rmb")
    points = runs[runs["batch_id"].isin(RMB_CAPACITY_BATCHES)].sort_values("trainable_params").reset_index(drop=True)
    if len(points) < 2:
        raise FileNotFoundError(f"runs_rmb.csv belum memuat batch kapasitas {RMB_CAPACITY_BATCHES}")
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(FIGURE_WIDTH_IN, 2.8), constrained_layout=True)
        ax.plot(points["trainable_params"], points["val_f1_macro"], color=BLUE_RAMP[4], linewidth=1.5, marker="o", markersize=5)
        scores = points["val_f1_macro"].to_numpy()
        for i, row in points.iterrows():
            # Puncak lokal diberi label di atas titik agar tidak menabrak label ruas yang turun di kanannya
            is_peak = 0 < i < len(points) - 1 and scores[i] > scores[i - 1] and scores[i] > scores[i + 1]
            ax.annotate(to_head_label(row["head_arch"], row["hidden_dim"]), (row["trainable_params"], row["val_f1_macro"]),
                        textcoords="offset points", xytext=(6, 5 if is_peak else -11), ha="left", fontsize=7.5,
                        color=MUTED_INK, bbox=LABEL_BOX)
            if i == 0:
                continue
            before = points.loc[i - 1]
            gain = (row["val_f1_macro"] - before["val_f1_macro"]) * 100
            per_100k = gain / (row["trainable_params"] - before["trainable_params"]) * 1e5
            # Label ruas di sisi atas garis: kiri untuk ruas naik, kanan untuk ruas turun
            ax.annotate(format_number(per_100k, 2, sign=True),
                        ((row["trainable_params"] + before["trainable_params"]) / 2,
                         (row["val_f1_macro"] + before["val_f1_macro"]) / 2),
                        textcoords="offset points", xytext=(-4, 5) if gain >= 0 else (4, 5),
                        ha="right" if gain >= 0 else "left", fontsize=7.5, color=INK, bbox=LABEL_BOX)
        ax.text(0.99, 0.03, "angka pada ruas: kenaikan marjinal, pp per 100.000 parameter",
                transform=ax.transAxes, ha="right", fontsize=7.5, color=MUTED_INK)
        ax.set_xlabel("trainable parameters")
        ax.set_ylabel("F1-macro validation")
        ax.xaxis.set_major_formatter(number_formatter(0))
        ax.yaxis.set_major_formatter(number_formatter(3))
        ax.grid(axis="y")
        margin = (points["val_f1_macro"].max() - points["val_f1_macro"].min()) * 0.25
        ax.set_ylim(points["val_f1_macro"].min() - margin, points["val_f1_macro"].max() + margin)
        return save_figure(fig, "gambar_4_2")


def plot_figure_4_3() -> list[Path]:
    """Heatmap divergen selisih RAC terhadap tanpa RAC, pada head juara RM-c."""
    final = load_best()["rmc"]["config"]
    grid = to_rmc_grid(load_runs("rmc"))
    pivot = grid[grid["rmb_run_id"] == int(final["rmb_run_id"])].pivot_table(index="alpha", columns="k", values="gain_pp", aggfunc="max")
    limit = max(float(np.nanmax(np.abs(pivot.values))), TIE_THRESHOLD_PP * 2)
    cmap = LinearSegmentedColormap.from_list("divergen", [WARM_GRAY, NEUTRAL, BLUE_RAMP[2], BLUE_RAMP[5]])
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(FIGURE_WIDTH_IN, 4.2), constrained_layout=True)
        image = ax.imshow(pivot.values, cmap=cmap, norm=TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit), aspect="auto")
        annotate_cells(ax, image, pivot.values, lambda v: format_number(v, 2, sign=True))
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                if abs(pivot.values[i, j]) <= TIE_THRESHOLD_PP + 1e-9:
                    ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, hatch="////", edgecolor="#b5b5b5", linewidth=0))
        frame_value(ax, pivot, final["alpha"], final["k"])
        ax.set_xticks(range(len(pivot.columns)), [str(int(c)) for c in pivot.columns])
        ax.set_yticks(range(len(pivot.index)), [format_number(a, 1) for a in pivot.index])
        ax.set_xlabel("k (jumlah tetangga)")
        ax.set_ylabel("α (bobot cabang retrieval)")
        clear_spines(ax)
        colorbar = fig.colorbar(image, ax=ax, fraction=0.05, pad=0.02)
        colorbar.set_label("selisih F1-macro validation terhadap tanpa RAC (pp)")
        colorbar.ax.yaxis.set_major_formatter(number_formatter(2))
        ax.legend(handles=[
            Patch(facecolor="white", edgecolor="#b5b5b5", hatch="////",
                  label=f"di dalam ambang seri ±{format_number(TIE_THRESHOLD_PP, 2)} pp"),
            Patch(facecolor="none", edgecolor=FINAL_ORANGE, linewidth=1.8, label="konfigurasi final"),
        ], loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
        return save_figure(fig, "gambar_4_3")

In [ ]:
def plot_figure_4_4() -> list[Path]:
    """Dumbbell kontribusi RAC per head RM-b: tanpa RAC, terbaik, dan bersama."""
    best = load_best()
    final, official_head = best["rmc"], int(best["rmb"]["run_id"])
    final_head = int(final["config"]["rmb_run_id"])
    rows = summarize_rac_per_head(load_runs("rmc"), load_runs("rmb")).sort_values("f1_no_rac").reset_index(drop=True)
    positions = np.arange(len(rows))
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(FIGURE_WIDTH_IN, 5.0), constrained_layout=True)
        ax.hlines(positions, rows["f1_no_rac"], rows["f1_best"], color=BLUE_RAMP[2], linewidth=1.2, zorder=1)
        ax.scatter(rows["f1_no_rac"], positions, s=22, color=BLUE_RAMP[1], edgecolor=BLUE_RAMP[4], linewidth=0.6,
                   zorder=2, label="tanpa RAC")
        ax.scatter(rows["f1_shared"], positions, s=26, facecolor="white", edgecolor=BLUE_RAMP[5], linewidth=1.0,
                   zorder=3, label="konfigurasi fusi bersama")
        ax.scatter(rows["f1_best"], positions, s=26, color=BLUE_RAMP[5], zorder=4, label="konfigurasi fusi terbaik per head")
        grid = to_rmc_grid(load_runs("rmc"))
        final_cell = grid[(grid["rmb_run_id"] == final_head) & np.isclose(grid["alpha"], float(final["config"]["alpha"]))
                          & (grid["k"] == int(final["config"]["k"]))]
        final_rows = rows[rows["rmb_run_id"] == final_head]
        ax.scatter(final_cell["val_f1_macro"].iloc[:1].tolist() * len(final_rows), final_rows.index, s=70,
                   facecolor="none", edgecolor=FINAL_ORANGE, linewidth=1.8, zorder=5, label="konfigurasi final")
        span = rows[["f1_no_rac", "f1_best"]].to_numpy()
        offset = (span.max() - span.min()) * 0.02
        for position, row in rows.iterrows():
            ax.text(max(row["f1_best"], row["f1_shared"]) + offset, position,
                    format_number(row["gain_best_pp"], 2, sign=True), va="center", fontsize=7, color=INK)
        labels = [f"#{int(row['rmb_run_id'])} {to_head_label(row['head_arch'], row['hidden_dim'])}" for _, row in rows.iterrows()]
        ax.set_yticks(positions, labels)
        for label, (_, row) in zip(ax.get_yticklabels(), rows.iterrows(), strict=True):
            if row["head_arch"] == "linear" or int(row["rmb_run_id"]) == official_head:
                label.set_fontweight("bold")
                label.set_color(INK)
        ax.set_ylim(len(rows) - 0.5, -0.5)
        ax.set_xlabel("F1-macro validation")
        ax.xaxis.set_major_formatter(number_formatter(3))
        ax.set_xlim(span.min() - offset * 2, span.max() + offset * 8)
        ax.grid(axis="x")
        ax.tick_params(axis="y", length=0, labelsize=7)
        ax.legend(loc="upper center", bbox_to_anchor=(0.45, -0.07), ncol=2)
        return save_figure(fig, "gambar_4_4")

## 5. Gambar 4.5-4.8: hasil benchmark final

- **4.5** Confusion matrix test ketiga skenario, dinormalisasi per kelas aktual.
- **4.6** Biaya pelatihan RM-b dan RM-c relatif terhadap RM-a, sumbu log. RM-c tidak melatih apa pun; biayanya adalah biaya head RM-b yang dipakai juaranya, diambil dari baris head itu di `runs_rmb.csv`.
- **4.7** Stacked bar dekomposisi latency inferensi per skenario.
- **4.8** Scatter F1-macro test terhadap training time dan inference latency; garis putus = ambang 3 pp di bawah RM-a.

In [ ]:
def plot_figure_4_5() -> list[Path]:
    """Confusion matrix test ketiga skenario, dinormalisasi per kelas aktual."""
    predictions = load_metrics_csv("final_predictions.csv")
    cmap = LinearSegmentedColormap.from_list("biru_nila", BLUE_RAMP)
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(1, 3, figsize=(FIGURE_WIDTH_IN, 2.3), constrained_layout=True)
        for ax, scenario in zip(axes, SCENARIO_LABELS, strict=True):
            counts = pd.crosstab(predictions["label"], predictions[f"pred_{scenario}"]).reindex(
                index=range(len(CLASS_NAMES)), columns=range(len(CLASS_NAMES)), fill_value=0)
            matrix = counts.to_numpy()
            share = matrix / matrix.sum(axis=1, keepdims=True)
            image = ax.imshow(share, cmap=cmap, vmin=0, vmax=1)
            for i in range(len(CLASS_NAMES)):
                for j in range(len(CLASS_NAMES)):
                    ax.text(j, i, f"{format_number(matrix[i, j], 0)}\n({format_number(share[i, j] * 100, 1)}%)",
                            ha="center", va="center", fontsize=7.5, color=pick_text_color(image.cmap(image.norm(share[i, j]))))
            ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES)
            ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES if ax is axes[0] else [])
            ax.set_xlabel("prediksi")
            ax.set_ylabel("aktual" if ax is axes[0] else "")
            ax.set_title(SCENARIO_LABELS[scenario])
            clear_spines(ax)
        colorbar = fig.colorbar(image, ax=list(axes), fraction=0.04, pad=0.02)
        colorbar.set_label("proporsi per kelas aktual (%)")
        colorbar.ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: format_number(v * 100, 0)))
        return save_figure(fig, "gambar_4_5")


def summarize_training_costs() -> pd.DataFrame:
    """Trainable params, training time, dan memori latih juara tiap skenario (RM-c = head yang dipakainya)."""
    best = load_best()
    rma = load_runs("rma").set_index("run_id").loc[int(best["rma"]["run_id"])]
    heads = load_runs("rmb").set_index("run_id")
    columns = ["trainable_params", "train_time_s", "train_peak_mem_mb"]
    costs = {"RM-a": {"trainable_params": float(rma["trainable_params"]), "train_time_s": float(rma["train_time_s"]),
                      "train_peak_mem_mb": float(rma["peak_mem_mb"])}}
    for label, run_id in [("RM-b", best["rmb"]["run_id"]), ("RM-c", best["rmc"]["config"]["rmb_run_id"])]:
        costs[label] = {column: float(heads.loc[int(run_id), column]) for column in columns}
    return pd.DataFrame(costs).T


def count_percent_decimals(value: float) -> int:
    """Jumlah desimal label persen: 0 untuk >= 10, selebihnya cukup untuk dua angka bermakna."""
    if value >= 10:
        return 0
    return max(1, -int(np.floor(np.log10(value))) + 1)


def plot_figure_4_6() -> list[Path]:
    """Biaya pelatihan RM-b dan RM-c relatif terhadap RM-a, sumbu log."""
    costs = summarize_training_costs()
    metrics = [("trainable_params", "trainable parameters"), ("train_time_s", "training time"),
               ("train_peak_mem_mb", "memori GPU pelatihan")]
    if (costs[[m for m, _ in metrics]] <= 0).any().any():
        raise ValueError("Biaya pelatihan harus positif untuk sumbu logaritmik")
    width = 0.26
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(FIGURE_WIDTH_IN, 3.0), constrained_layout=True)
        for offset, scenario in zip((-width, 0.0, width), costs.index, strict=True):
            relative = [costs.loc[scenario, m] / costs.loc["RM-a", m] * 100 for m, _ in metrics]
            bars = ax.bar(np.arange(len(metrics)) + offset, relative, width=width * 0.92, color=SCENARIO_COLORS[scenario],
                          label=scenario, hatch="////" if scenario == "RM-c" else None,
                          edgecolor=BLUE_RAMP[4] if scenario == "RM-c" else "white", linewidth=0.5)
            for bar, value in zip(bars, relative, strict=True):
                ax.text(bar.get_x() + bar.get_width() / 2, value * 1.15,
                        f"{format_number(value, count_percent_decimals(value))}%", ha="center", va="bottom", fontsize=7, color=INK)
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(percent_log_formatter())
        ax.set_ylim(top=100 * 8)
        ax.set_xticks(range(len(metrics)), [label for _, label in metrics])
        ax.set_ylabel("persentase terhadap RM-a (skala log)")
        ax.grid(axis="y", which="major")
        ax.legend(handles=[
            Patch(facecolor=SCENARIO_COLORS["RM-a"], label="RM-a"),
            Patch(facecolor=SCENARIO_COLORS["RM-b"], label="RM-b"),
            Patch(facecolor=SCENARIO_COLORS["RM-c"], hatch="////", edgecolor=BLUE_RAMP[4], label="RM-c (diwarisi dari head RM-b)"),
        ], loc="upper right", ncol=3)
        return save_figure(fig, "gambar_4_6")

In [ ]:
def plot_figure_4_7() -> list[Path]:
    """Stacked bar dekomposisi latency inferensi per skenario."""
    table = (load_metrics_csv("latency_breakdown.csv")
             .pivot_table(index="scenario", columns="component", values="latency_ms")
             .reindex(index=["RM-a", "RM-b", "RM-c"], columns=LATENCY_COMPONENTS).fillna(0.0))
    colors = [BLUE_RAMP[4], BLUE_RAMP[2], BLUE_RAMP[1], BLUE_RAMP[0]]
    hatches = [None, None, "....", "////"]
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(FIGURE_WIDTH_IN, 3.0), constrained_layout=True)
        bottom = np.zeros(len(table))
        for component, color, hatch in zip(LATENCY_COMPONENTS, colors, hatches, strict=True):
            values = table[component].to_numpy()
            ax.bar(table.index, values, bottom=bottom, width=0.5, color=color, hatch=hatch,
                   edgecolor="white", linewidth=0.8, label=component)
            bottom += values
        totals = table.sum(axis=1)
        for position, (scenario, total) in enumerate(totals.items()):
            ax.text(position, total * 1.02, f"{format_number(total, 2)} ms", ha="center", va="bottom", fontsize=7.5, color=INK)
            share = table.loc[scenario, "encoder"] / total * 100
            ax.text(position, table.loc[scenario, "encoder"] / 2, f"encoder\n{format_number(share, 1)}%",
                    ha="center", va="center", fontsize=7, color="white")
        ax.set_ylim(0, totals.max() * 1.15)
        ax.set_ylabel("latency per sampel (ms)")
        ax.yaxis.set_major_formatter(number_formatter(1))
        ax.grid(axis="y")
        ax.legend(loc="upper left", ncol=4)
        return save_figure(fig, "gambar_4_7")


def plot_figure_4_8() -> list[Path]:
    """Scatter F1-macro test terhadap training time dan inference latency."""
    comparison = load_metrics_csv("final_comparison.csv").set_index("model")
    threshold = comparison.loc["RM-a", "test_f1_macro"] - MAX_F1_GAP_PP / 100
    with plt.rc_context(STYLE):
        fig, (left, right) = plt.subplots(1, 2, figsize=(FIGURE_WIDTH_IN, 2.6), sharey=True, constrained_layout=True)
        for ax, column, label in [(left, "train_time_s", "training time (s, skala log)"),
                                  (right, "infer_latency_ms", "inference latency (ms)")]:
            for scenario, row in comparison.iterrows():
                ax.scatter(row[column], row["test_f1_macro"], s=36, color=SCENARIO_COLORS[scenario],
                           marker=SCENARIO_MARKERS[scenario], edgecolor="white", linewidth=0.6, zorder=3)
            # Skenario dengan titik yang sama persis (mis. RM-b dan RM-c pada training time) diberi satu label gabungan
            for (x, y), group in comparison.groupby([column, "test_f1_macro"], sort=False):
                ax.annotate(" = ".join(group.index), (x, y), textcoords="offset points",
                            xytext=(5, 4), fontsize=7.5, color=INK)
            ax.axhline(threshold, color=MUTED_INK, linestyle="--", linewidth=0.8)
            ax.set_xlabel(label)
            ax.grid(alpha=1.0)
        left.set_xscale("log")
        left.set_xlim(comparison["train_time_s"].min() / 2, comparison["train_time_s"].max() * 3)
        left.xaxis.set_major_locator(LogLocator(base=10, subs=(1.0, 2.0, 5.0)))
        left.xaxis.set_minor_formatter(NullFormatter())
        left.xaxis.set_major_formatter(number_formatter(0))
        # Mulai dari nol: ketiga skenario TIDAK bergeser di fase inferensi, sumbu terpotong membesar-besarkan selisih kecil
        right.set_xlim(0, comparison["infer_latency_ms"].max() * 1.3)
        right.xaxis.set_major_formatter(number_formatter(1))
        left.yaxis.set_major_formatter(number_formatter(3))
        left.set_ylabel("F1-macro test")
        left.text(0.02, threshold, " ambang 3 pp di bawah RM-a", transform=left.get_yaxis_transform(),
                  va="bottom", fontsize=7, color=MUTED_INK)
        low, high = min(threshold, comparison["test_f1_macro"].min()), comparison["test_f1_macro"].max()
        left.set_ylim(low - (high - low) * 0.15, high + (high - low) * 0.15)
        return save_figure(fig, "gambar_4_8")

## 6. Bangkitkan gambar

In [ ]:
def run_builders(builders: dict) -> dict:
    """Jalankan tiap pembangun; yang lognya belum ada dicatat sebagai `dilewati: ...`."""
    results = {}
    for number, build in builders.items():
        try:
            results[number] = build()
        except FileNotFoundError as error:
            results[number] = f"dilewati: {error}"
    return results


figure_results = run_builders({
    "4.1": plot_figure_4_1, "4.2": plot_figure_4_2, "4.3": plot_figure_4_3, "4.4": plot_figure_4_4,
    "4.5": plot_figure_4_5, "4.6": plot_figure_4_6, "4.7": plot_figure_4_7, "4.8": plot_figure_4_8,
})
for number, output in figure_results.items():
    print(f"Gambar {number}: {output if isinstance(output, str) else ', '.join(p.name for p in output)}")
    if isinstance(output, list):
        display(Image(filename=output[0], width=700))

## 7. Data Tabel 4.1-4.4: lingkungan dan data

Tiap pengumpul mengembalikan dict `number`, `title`, `frame` (nilai presisi penuh), `sources` (berkas yang dibaca), dan `notes` (konteks untuk menyusun tabel).

In [ ]:
def to_table(number: str, title: str, frame: pd.DataFrame, sources: list[str], notes: list[str] | None = None) -> dict:
    """Data mentah satu tabel."""
    return {"number": number, "title": title, "frame": frame, "sources": sources, "notes": notes or []}


def compute_reduction_pct(value: float | None, baseline: float | None) -> float | None:
    """Reduksi terhadap baseline dalam persen; `None` bila salah satunya tidak ada."""
    if value is None or baseline is None or pd.isna(value) or pd.isna(baseline) or not baseline:
        return None
    return (1.0 - float(value) / float(baseline)) * 100


def collect_table_4_1() -> dict:
    """Spesifikasi lingkungan eksperimen dan status sesi."""
    hardware = load_required_json(OUT_DIR / "hardware.json")
    keys = ["gpu", "vram_total_mb", "driver", "cuda_version", "cpu", "cpu_logical_cores", "ram_total_gb", "os",
            "python", "torch", "transformers", "faiss", "seed"]
    rows = [{"komponen": key, "nilai": hardware.get(key)} for key in keys]
    for i, session in enumerate(hardware.get("tuning_sessions", []), start=1):
        rows.append({"komponen": f"sesi_tuning_{i}_recorded_at", "nilai": session["recorded_at"]})
        rows.append({"komponen": f"sesi_tuning_{i}_boot_time", "nilai": session["boot_time"]})
    final = hardware.get("final_session")
    if final:
        rows += [{"komponen": "sesi_final_recorded_at", "nilai": final["recorded_at"]},
                 {"komponen": "sesi_final_boot_time", "nilai": final["boot_time"]},
                 {"komponen": "sesi_final_same_boot_as_tuning", "nilai": final.get("same_boot_as_tuning")}]
    notes = ["Status sesi: satu sesi mesin bila hanya ada satu sesi tuning dan sesi_final_same_boot_as_tuning=True; "
             "selain itu hardware sama tetapi sesi mesin berbeda."]
    return to_table("4.1", "Spesifikasi lingkungan eksperimen", pd.DataFrame(rows), ["hardware.json"], notes)


def collect_table_4_2() -> dict:
    """Jumlah baris per tahap pra-pemrosesan dan baris terbuang per kelas di guard."""
    counts = load_required_json(METADATA_PATH)["counts"]
    stats = load_required_json(STATS_PATH)
    stages = [("mentah", counts["raw"]), ("buang baris kosong", counts["after_missing"]),
              ("deduplikasi NFKC", counts["after_dedup"]), ("guard anti-leakage", counts["final_total"])]
    rows, previous = [], None
    for name, count in stages:
        change = None if previous is None else count - previous
        rows.append({"tahap": name, "jumlah_baris": count, "perubahan_baris": change,
                     "perubahan_pct": None if previous is None else change / previous * 100})
        previous = count
    removed = stats.get("leakage_removed_by_class", {})
    rows.append({"tahap": "guard: terbuang kelas judi", "jumlah_baris": removed.get("judi", 0)})
    rows.append({"tahap": "guard: terbuang kelas non-judi", "jumlah_baris": removed.get("non_judi", 0)})
    return to_table("4.2", "Reduksi jumlah baris per tahap pra-pemrosesan", pd.DataFrame(rows),
                    ["data/processed/metadata.json", "outputs/preprocessing/preprocessing_stats.json"])


def collect_table_4_3() -> dict:
    """Persentase baris ber-[UNK] sebelum dan sesudah NFKC, per kelas."""
    nfkc = load_required_json(STATS_PATH)["nfkc_unk"]
    rates = nfkc["rates_pct"]
    rows = [{"kondisi": condition, "unk_judi_pct": rates[condition]["judi"], "unk_non_judi_pct": rates[condition]["non_judi"]}
            for condition in ["sebelum_nfkc", "sesudah_nfkc"]]
    notes = [f"{nfkc['unit']}; tokenizer pelatihan (memuat [URL], [MENTION], [NUM]); sebelum NFKC = pembersihan "
             f"yang sama tanpa langkah NFKC; dataset final {nfkc['n_rows']['judi']} baris judi, "
             f"{nfkc['n_rows']['non_judi']} baris non-judi."]
    return to_table("4.3", "Dampak normalisasi NFKC", pd.DataFrame(rows), ["outputs/preprocessing/preprocessing_stats.json"], notes)


def collect_table_4_4() -> dict:
    """Komposisi dataset final, class weight, dan konflik label."""
    metadata = load_required_json(METADATA_PATH)
    frame = pd.DataFrame([{"split": name, "jumlah_baris": split["n"], "kelas_0": split["L0"], "kelas_1": split["L1"],
                           "persen_kelas_1": split["L1"] / split["n"] * 100}
                          for name, split in ((name, metadata["splits"][name]) for name in ["train", "val", "test"])])
    total = frame[["jumlah_baris", "kelas_0", "kelas_1"]].sum()
    frame.loc[len(frame)] = {"split": "total", **total.to_dict(), "persen_kelas_1": total["kelas_1"] / total["jumlah_baris"] * 100}
    weights = metadata["class_weights"]
    frame["class_weight"] = [f"0: {weights['0']}; 1: {weights['1']}" if s == "train" else None for s in frame["split"]]
    conflict = metadata.get("label_conflict", {})
    notes = [f"Class weight dari train. Konflik label: {conflict.get('groups')} kelompok, "
             f"{conflict.get('rows')} baris, kebijakan {conflict.get('policy')}."]
    return to_table("4.4", "Dataset final", frame, ["data/processed/metadata.json"], notes)

## 8. Data Tabel 4.5-4.7: ruang pencarian yang dieksekusi

In [ ]:
def summarize_search_space(number: str, title: str, runs: pd.DataFrame, stage_labels: dict,
                           params: list[str], source: str) -> dict:
    """Nilai yang dicoba per tahap, jumlah konfigurasi unik, jumlah eksekusi, dan parameter yang tetap."""
    runs = runs.sort_values("run_id").copy()
    runs["_stage"] = runs["batch_id"].fillna("").astype(str).map(lambda batch: stage_labels.get(batch, batch or "run tunggal"))
    rows = []
    for stage in dict.fromkeys(runs["_stage"]):
        subset = runs[runs["_stage"] == stage]
        for param in params:
            values = sorted(subset[param].dropna().unique().tolist(), key=lambda v: (str(type(v)), v))
            if len(values) > 1:
                rows.append({"tahap": stage, "parameter": param, "nilai": "; ".join(map(str, values)), "jumlah": len(values)})
        rows.append({"tahap": stage, "parameter": "konfigurasi_unik", "nilai": "", "jumlah": len(subset[params].drop_duplicates())})
        rows.append({"tahap": stage, "parameter": "eksekusi", "nilai": "", "jumlah": len(subset)})
    rows.append({"tahap": "total", "parameter": "konfigurasi_unik", "nilai": "", "jumlah": len(runs[params].drop_duplicates())})
    rows.append({"tahap": "total", "parameter": "eksekusi", "nilai": "", "jumlah": len(runs)})
    for param in params:
        if runs[param].dropna().nunique() == 1:
            rows.append({"tahap": "tetap di seluruh tahap", "parameter": param,
                         "nilai": str(runs[param].dropna().iloc[0]), "jumlah": 1})
    return to_table(number, title, pd.DataFrame(rows), [source])


def collect_table_4_5() -> dict:
    """Ruang pencarian RM-a yang benar-benar dieksekusi."""
    return summarize_search_space("4.5", "Ruang pencarian hyperparameter RM-a", load_runs("rma"), RMA_STAGES,
                                  ["lr", "epochs", "batch", "warmup_ratio", "weight_decay", "micro_batch"], "runs_rma.csv")


def collect_table_4_6() -> dict:
    """Ruang pencarian RM-b yang benar-benar dieksekusi."""
    runs = load_runs("rmb")
    runs["head"] = [to_head_label(a, h) for a, h in zip(runs["head_arch"], runs["hidden_dim"], strict=True)]
    return summarize_search_space("4.6", "Ruang pencarian hyperparameter RM-b", runs, RMB_STAGES,
                                  ["head", "lr", "epochs", "dropout", "weight_decay", "batch"], "runs_rmb.csv")


def collect_table_4_7() -> dict:
    """Ruang pencarian RM-c, jumlah evaluasi, dan indeks FAISS."""
    runs = load_runs("rmc")
    table = summarize_search_space("4.7", "Ruang pencarian RM-c", runs, RMC_STAGES,
                                   ["rmb_run_id", "alpha", "k", "weighting"], "runs_rmc.csv")
    train_rows = load_required_json(METADATA_PATH)["splits"]["train"]["n"]
    vectors = sorted(int(v) for v in runs["index_vectors"].dropna().unique())
    index_types = sorted(str(v) for v in runs["index_type"].dropna().unique()) if "index_type" in runs else []
    extra = pd.DataFrame([
        {"tahap": "indeks FAISS", "parameter": "vektor_dalam_indeks", "nilai": "; ".join(map(str, vectors)), "jumlah": len(vectors)},
        {"tahap": "indeks FAISS", "parameter": "jumlah_baris_train", "nilai": str(train_rows), "jumlah": 1},
        {"tahap": "indeks FAISS", "parameter": "sama_dengan_train", "nilai": str(vectors == [train_rows]), "jumlah": 1},
        {"tahap": "indeks FAISS", "parameter": "jenis_indeks", "nilai": "; ".join(index_types), "jumlah": len(index_types)},
    ])
    table["frame"] = pd.concat([table["frame"], extra], ignore_index=True)
    table["sources"].append("data/processed/metadata.json")
    return table

## 9. Data Tabel 4.8-4.12: hasil tuning pada validation

In [ ]:
def summarize_top_runs(number: str, title: str, scenario: str, columns: list[str]) -> dict:
    """`TOP_N` run teratas satu skenario beserta selisih dan status seri terhadap juara."""
    runs = load_runs(scenario)
    champion_id = int(load_best()[scenario]["run_id"])
    champion_f1 = float(runs.loc[runs["run_id"] == champion_id, "val_f1_macro"].iloc[0])
    top = rank_runs(runs, scenario).head(TOP_N)
    frame = top[[c for c in columns if c in top.columns]].copy()
    frame.insert(0, "peringkat", range(1, len(frame) + 1))
    frame["selisih_terhadap_juara_pp"] = (frame["val_f1_macro"] - champion_f1) * 100
    frame["status"] = ["juara" if int(run_id) == champion_id else ("seri" if abs(delta) <= TIE_THRESHOLD_PP + 1e-9 else "tidak seri")
                       for run_id, delta in zip(frame["run_id"], frame["selisih_terhadap_juara_pp"], strict=True)]
    notes = [f"Peringkat: F1-macro, lalu F1 judi, lalu training time. Seri = selisih <= {TIE_THRESHOLD_PP} pp dari juara."]
    return to_table(number, title, frame.reset_index(drop=True), [f"runs_{scenario}.csv", "best.json"], notes)


def collect_table_4_8() -> dict:
    """Tiga konfigurasi terbaik RM-a pada validation."""
    columns = ["run_id", "lr", "epochs", "batch", "warmup_ratio", "weight_decay", "best_epoch", "val_f1_macro",
               "val_precision_macro", "val_recall_macro", "val_f1_judi", "train_time_s"]
    return summarize_top_runs("4.8", "Tiga konfigurasi terbaik RM-a pada validation", "rma", columns)


def collect_table_4_9() -> dict:
    """Kapasitas classification head terhadap F1 RM-b."""
    runs = load_runs("rmb")
    points = runs[runs["batch_id"].isin(RMB_CAPACITY_BATCHES)].sort_values("trainable_params").reset_index(drop=True)
    if points.empty:
        raise FileNotFoundError("runs_rmb.csv belum memuat tahap arsitektur head")
    frame = pd.DataFrame({
        "run_id": points["run_id"],
        "arsitektur_head": [to_head_label(a, h) for a, h in zip(points["head_arch"], points["hidden_dim"], strict=True)],
        "trainable_params": points["trainable_params"],
        "val_f1_macro": points["val_f1_macro"],
    })
    frame["kenaikan_marjinal_pp_per_100rb_param"] = frame["val_f1_macro"].diff() * 100 / frame["trainable_params"].diff() * 1e5
    return to_table("4.9", "Kapasitas classification head terhadap F1 RM-b", frame, ["runs_rmb.csv"],
                    ["Kenaikan marjinal terhadap arsitektur sebelumnya pada urutan trainable params."])


def collect_table_4_10() -> dict:
    """Tiga konfigurasi terbaik RM-b pada validation."""
    columns = ["run_id", "head_arch", "hidden_dim", "lr", "epochs", "dropout", "weight_decay", "batch", "best_epoch",
               "val_f1_macro", "val_f1_judi", "train_time_s"]
    table = summarize_top_runs("4.10", "Tiga konfigurasi terbaik RM-b pada validation", "rmb", columns)
    if (OUT_DIR / "candidates.json").exists():
        excluded = load_candidates(OUT_DIR / "candidates.json")["scenarios"]["rmb"].get("excluded_epoch_variants", [])
        table["notes"].append("Varian anggaran epoch dengan epoch terbaik sama dengan juara (bobot identik, learning rate RM-b "
                              f"konstan): run {excluded if excluded else 'tidak ada'}.")
        table["sources"].append("candidates.json")
    return table

In [ ]:
def collect_table_4_11() -> dict:
    """Tiga konfigurasi terbaik RM-c, default, penantang, final, dan bootstrap."""
    runs = load_runs("rmc")
    decision = load_required_json(OUT_DIR / "rmc_champion_decision.json")
    baseline = to_rmc_grid(runs).groupby("rmb_run_id")["f1_no_rac"].first()
    roles = {}
    for rank, run_id in enumerate(rank_rmc_runs(runs)["run_id"].head(TOP_N), start=1):
        roles.setdefault(int(run_id), []).append(f"peringkat {rank}")
    roles.setdefault(int(decision["challenger"]["run_id"]), []).append("penantang")
    roles.setdefault(int(decision["default"]["run_id"]), []).append("default (head RM-b resmi)")
    roles.setdefault(int(load_best()["rmc"]["run_id"]), []).append("final")
    rows = []
    for run_id, role in roles.items():
        row = runs[runs["run_id"] == run_id].iloc[0]
        no_rac = float(baseline[int(row["rmb_run_id"])])
        rows.append({"peran": ", ".join(role), "run_id": run_id, "head_rmb_run_id": int(row["rmb_run_id"]),
                     "alpha": row["alpha"], "k": int(row["k"]), "weighting": row["weighting"],
                     "val_f1_macro": row["val_f1_macro"], "val_f1_judi": row["val_f1_judi"],
                     "val_f1_tanpa_rac": no_rac, "selisih_terhadap_tanpa_rac_pp": (row["val_f1_macro"] - no_rac) * 100})
    bootstrap = decision.get("bootstrap") or {}
    notes = [
        decision["rule"],
        f"putusan: {decision['winner']}; head resmi RM-b #{decision['official_rmb_run_id']}; "
        f"head_is_official_rmb={decision['head_is_official_rmb']}",
        (f"bootstrap: observed_delta_pp={bootstrap.get('observed_delta_pp')}, mean_delta_pp={bootstrap.get('mean_delta_pp')}, "
         f"ci95_pp=[{bootstrap.get('ci_low_pp')}, {bootstrap.get('ci_high_pp')}], n_boot={bootstrap.get('n_boot')}, "
         f"seed={bootstrap.get('seed')}, n_samples={bootstrap.get('n_samples')}")
        if bootstrap else "penantang sama dengan default; bootstrap tidak diperlukan",
        decision["cost_note"],
    ]
    return to_table("4.11", "Tiga konfigurasi terbaik RM-c pada validation", pd.DataFrame(rows),
                    ["runs_rmc.csv", "rmc_champion_decision.json", "best.json"], notes)


def collect_table_4_12() -> dict:
    """Kontribusi RAC per kelompok head (kelompok menurut arsitektur)."""
    summary = summarize_rac_per_head(load_runs("rmc"), load_runs("rmb"))
    summary["kelompok"] = [to_head_group(a, h) for a, h in zip(summary["head_arch"], summary["hidden_dim"], strict=True)]
    rows = [{
        "kelompok_head": group, "jumlah_head": len(subset),
        "f1_tanpa_rac_min": subset["f1_no_rac"].min(), "f1_tanpa_rac_maks": subset["f1_no_rac"].max(),
        "f1_dengan_rac_min": subset["f1_best"].min(), "f1_dengan_rac_maks": subset["f1_best"].max(),
        "median_gain_terbaik_pp": subset["gain_best_pp"].median(),
        "median_gain_konfigurasi_bersama_pp": subset["gain_shared_pp"].median(),
        "median_gain_f1_judi_pp": subset["gain_judi_best_pp"].median(),
    } for group, subset in summary.groupby("kelompok", sort=False)]
    order = {"Linear": 0, "MLP hidden dim < 1024": 1}
    frame = pd.DataFrame(rows).sort_values("kelompok_head", key=lambda s: s.map(lambda g: order.get(g, 2)))
    notes = [
        "Kelompok ditetapkan menurut arsitektur head, bukan F1. Kelompok berisi 1 head: tampilkan nilainya langsung.",
        "F1 tanpa RAC = baris alpha = 0 di grid RM-c (head dimuat dari checkpoint).",
        f"Konfigurasi bersama (rata-rata gain tertinggi di seluruh head): alpha={summary['shared_alpha'].iloc[0]}, "
        f"k={summary['shared_k'].iloc[0]}.",
    ]
    return to_table("4.12", "Kontribusi RAC per kelompok head RM-b", frame.reset_index(drop=True),
                    ["runs_rmc.csv", "runs_rmb.csv"], notes)

## 10. Data Tabel 4.13-4.16: perbandingan final

In [ ]:
def collect_table_4_13() -> dict:
    """Metrik validation konfigurasi final ketiga skenario."""
    rows = {s: load_champion_row(s) for s in SCENARIO_LABELS}
    params = {"rma": rows["rma"]["trainable_params"], "rmb": rows["rmb"]["trainable_params"],
              "rmc": rows["rmc"]["head_trainable_params"]}
    keys = ["val_f1_macro", "val_acc", "val_precision_macro", "val_recall_macro", "val_f1_judi"]
    frame = pd.DataFrame(
        [{"metrik": "run_id", **{SCENARIO_LABELS[s]: int(r["run_id"]) for s, r in rows.items()}},
         {"metrik": "konfigurasi", **{SCENARIO_LABELS[s]: str(load_best()[s]["config"]) for s in rows}},
         {"metrik": "trainable_params", **{SCENARIO_LABELS[s]: int(v) for s, v in params.items()}}]
        + [{"metrik": key, **{SCENARIO_LABELS[s]: float(r[key]) for s, r in rows.items()}} for key in keys]
        + [{"metrik": "selisih_f1_macro_terhadap_rma_pp",
            **{SCENARIO_LABELS[s]: None if s == "rma" else (float(r["val_f1_macro"]) - float(rows["rma"]["val_f1_macro"])) * 100
               for s, r in rows.items()}}]
    )
    notes = ["trainable_params RM-c = milik head RM-b yang dipakainya; RM-c sendiri tidak melatih apa pun."]
    return to_table("4.13", "Perbandingan konfigurasi final ketiga skenario pada validation", frame,
                    ["best.json", "runs_rma.csv", "runs_rmb.csv", "runs_rmc.csv"], notes)


def collect_table_4_14() -> dict:
    """Metrik test dan confusion matrix ketiga skenario."""
    results = compute_test_results()
    keys = ["f1_macro", "accuracy", "precision_macro", "recall_macro", "precision_class1", "recall_class1",
            "f1_class1", "tp", "fp", "fn", "tn"]
    frame = pd.DataFrame(
        [{"metrik": key, **{SCENARIO_LABELS[s]: r[key] for s, r in results.items()}} for key in keys]
        + [{"metrik": "selisih_f1_macro_terhadap_rma_pp",
            **{SCENARIO_LABELS[s]: None if s == "rma" else (r["f1_macro"] - results["rma"]["f1_macro"]) * 100
               for s, r in results.items()}}]
    )
    return to_table("4.14", "Hasil klasifikasi pada test set", frame, ["metrics/final_predictions.csv"],
                    ["class1 = kelas judi; TP/FP/FN/TN dengan kelas judi sebagai positif."])


def collect_table_4_15() -> dict:
    """F1 validation dan test serta selisihnya."""
    results = compute_test_results()
    val = {s: float(load_champion_row(s)["val_f1_macro"]) for s in SCENARIO_LABELS}
    rows = [{
        "skenario": label, "val_f1_macro": val[s], "test_f1_macro": results[s]["f1_macro"],
        "perubahan_val_ke_test_pp": (results[s]["f1_macro"] - val[s]) * 100,
        "selisih_terhadap_rma_val_pp": None if s == "rma" else (val[s] - val["rma"]) * 100,
        "selisih_terhadap_rma_test_pp": None if s == "rma" else (results[s]["f1_macro"] - results["rma"]["f1_macro"]) * 100,
    } for s, label in SCENARIO_LABELS.items()]
    return to_table("4.15", "Selisih validation ke test", pd.DataFrame(rows),
                    ["best.json", "runs_*.csv", "metrics/final_predictions.csv"])


def collect_table_4_16() -> dict:
    """Kandidat #1 dan #2 per skenario pada validation dan test.

    Raises:
        ValueError: Bila hash kandidat di `candidates_test.csv` tidak cocok dengan `candidates.json`.
    """
    candidates = load_candidates(OUT_DIR / "candidates.json")
    tested = load_metrics_csv("candidates_test.csv")
    if set(tested["candidates_sha256"].astype(str)) != {candidates["content_sha256"]}:
        raise ValueError("Hash kandidat di candidates_test.csv tidak cocok dengan candidates.json")
    rows = []
    for scenario, label in SCENARIO_LABELS.items():
        entry = candidates["scenarios"][scenario]
        second = entry.get("second")
        if not second:
            rows.append({"skenario": label, "perbedaan_kandidat_2": entry.get("note")})
            continue
        test = tested[tested["scenario"] == scenario].set_index("rank")["test_f1_macro"]
        rows.append({
            "skenario": label, "perbedaan_kandidat_2": str(entry["differs_in"]),
            "run_id_1": entry["first"]["run_id"], "run_id_2": second["run_id"],
            "val_f1_1": entry["first"]["val_f1_macro"], "val_f1_2": second["val_f1_macro"],
            "test_f1_1": float(test[1]), "test_f1_2": float(test[2]),
            "sebaran_test_pp": abs(float(test[1]) - float(test[2])) * 100,
        })
    notes = [candidates["rule"],
             f"ditetapkan {candidates['created_at']}; sha256 {candidates['content_sha256']}; hash di candidates_test.csv cocok."]
    return to_table("4.16", "Sensitivitas terhadap konfigurasi terpilih", pd.DataFrame(rows),
                    ["candidates.json", "metrics/candidates_test.csv"], notes)

## 11. Data Tabel 4.17-4.19 dan L.1: efisiensi dan putusan

Kriteria sukses juga dihitung terhadap RM-a TERMURAH (kasus terburuk bagi strategi ringan), bukan hanya terhadap juara RM-a.

In [ ]:
def summarize_efficiency() -> dict:
    """Angka efisiensi per skenario; RM-b/RM-c memakai biaya head, RM-c ditambah indeks FAISS."""
    best = load_best()
    rma = load_champion_row("rma")
    heads = load_runs("rmb").set_index("run_id")
    benchmark = load_metrics_csv("inference_benchmark.csv").set_index("scenario")
    index = load_required_json(OUT_DIR / "metrics" / "index_stats.json")

    def get_head_costs(run_id: int) -> dict:
        row = heads.loc[run_id]
        return {"head_rmb_run_id": run_id, "trainable_params": float(row["trainable_params"]),
                "train_time_s": float(row["train_time_s"]), "extract_time_s": float(row["extract_time_s"]),
                "head_train_time_s": float(row["head_train_time_s"]), "train_peak_mem_mb": float(row["train_peak_mem_mb"])}

    values = {
        "rma": {"head_rmb_run_id": None, "trainable_params": float(rma["trainable_params"]),
                "train_time_s": float(rma["train_time_s"]), "extract_time_s": None, "head_train_time_s": None,
                "train_peak_mem_mb": float(rma["peak_mem_mb"]), "index_build_s": None, "index_size_mb": None},
        "rmb": {**get_head_costs(int(best["rmb"]["run_id"])), "index_build_s": None, "index_size_mb": None},
        "rmc": {**get_head_costs(int(best["rmc"]["config"]["rmb_run_id"])),
                "index_build_s": float(index["build_time_s"]), "index_size_mb": float(index["size_mb"])},
    }
    for scenario, label in SCENARIO_LABELS.items():
        for key in ["infer_peak_gpu_mem_mb", "infer_latency_ms", "latency_warmup_runs", "latency_runs"]:
            values[scenario][key] = benchmark.loc[label, key] if key in benchmark.columns else None
    values["_index"] = index
    return values


def collect_table_4_17() -> dict:
    """Biaya pelatihan dan inferensi ketiga skenario serta reduksinya terhadap RM-a."""
    values = summarize_efficiency()
    keys = ["head_rmb_run_id", "trainable_params", "train_time_s", "extract_time_s", "head_train_time_s", "index_build_s",
            "train_peak_mem_mb", "infer_peak_gpu_mem_mb", "index_size_mb", "infer_latency_ms", "latency_warmup_runs", "latency_runs"]
    rows = []
    for key in keys:
        row = {"metrik": key, **{SCENARIO_LABELS[s]: values[s][key] for s in SCENARIO_LABELS}}
        if key not in ["head_rmb_run_id", "latency_warmup_runs", "latency_runs"]:
            row["reduksi_rmb_pct"] = compute_reduction_pct(values["rmb"][key], values["rma"][key])
            row["reduksi_rmc_pct"] = compute_reduction_pct(values["rmc"][key], values["rma"][key])
        rows.append(row)
    index = values["_index"]
    notes = [
        "train_time_s RM-b/RM-c = extract_time_s + head_train_time_s; RM-c memakai head hasil putusan juara "
        "(head_rmb_run_id), index_build_s dicatat terpisah.",
        "train_peak_mem_mb RM-b/RM-c termasuk ekstraksi fitur (maks ekstraksi dan latih head).",
        "Latency = jumlah komponen dari satu jalur yang sama, rata-rata latency_runs pengulangan setelah "
        "latency_warmup_runs pemanasan.",
        f"Indeks {index.get('index_type')}: {index.get('vectors')} vektor, dimensi {index.get('dimension')}, dari split {index.get('source_split')}.",
        "Biaya latih dari sesi tuning, biaya inferensi dari sesi benchmark final, hardware sama (Tabel 4.1). "
        "Reduksi negatif = lebih besar dari RM-a.",
    ]
    return to_table("4.17", "Perbandingan efisiensi komputasi", pd.DataFrame(rows),
                    ["best.json", "runs_rma.csv", "runs_rmb.csv", "metrics/inference_benchmark.csv", "metrics/index_stats.json"], notes)


def evaluate_criteria() -> dict:
    """Kriteria sukses RM-b dan RM-c terhadap juara RM-a, termasuk terhadap RM-a termurah."""
    results = compute_test_results()
    values = summarize_efficiency()
    cheapest = load_runs("rma").sort_values(["train_time_s", "run_id"]).iloc[0]
    outcome = {"_cheapest_rma": cheapest.to_dict()}
    for scenario in ["rmb", "rmc"]:
        delta = (results[scenario]["f1_macro"] - results["rma"]["f1_macro"]) * 100
        params = compute_reduction_pct(values[scenario]["trainable_params"], values["rma"]["trainable_params"])
        time_red = compute_reduction_pct(values[scenario]["train_time_s"], values["rma"]["train_time_s"])
        worst = compute_reduction_pct(values[scenario]["train_time_s"], float(cheapest["train_time_s"]))
        passed = [delta >= -MAX_F1_GAP_PP, params >= MIN_PARAM_REDUCTION_PCT, time_red >= MIN_TIME_REDUCTION_PCT]
        outcome[scenario] = {
            "selisih_f1_test_terhadap_rma_pp": delta,
            "reduksi_trainable_params_pct": params,
            "reduksi_training_time_pct": time_red,
            "reduksi_training_time_terhadap_rma_termurah_pct": worst,
            "jarak_ambang_f1_pp": delta + MAX_F1_GAP_PP,
            "jarak_ambang_params_pp": params - MIN_PARAM_REDUCTION_PCT,
            "jarak_ambang_training_time_pp": time_red - MIN_TIME_REDUCTION_PCT,
            "jarak_ambang_training_time_termurah_pp": worst - MIN_TIME_REDUCTION_PCT,
            "kriteria_terpenuhi": sum(passed),
            "kompetitif": sum(passed) >= MIN_CRITERIA,
        }
    return outcome


def collect_table_4_18() -> dict:
    """Evaluasi kriteria sukses."""
    outcome = evaluate_criteria()
    cheapest = outcome.pop("_cheapest_rma")
    frame = pd.DataFrame([{"skenario": SCENARIO_LABELS[s], **item} for s, item in outcome.items()])
    notes = [
        f"Ambang: selisih F1 >= -{MAX_F1_GAP_PP} pp, reduksi params >= {MIN_PARAM_REDUCTION_PCT}%, reduksi "
        f"training time >= {MIN_TIME_REDUCTION_PCT}% terhadap juara RM-a; kompetitif bila >= {MIN_CRITERIA} dari 3. "
        "Jarak ke ambang positif = lolos.",
        f"RM-a termurah (kasus terburuk): run #{int(cheapest['run_id'])}, lr={cheapest['lr']}, "
        f"epochs={cheapest['epochs']}, batch={cheapest['batch']}, warmup_ratio={cheapest['warmup_ratio']}, "
        f"weight_decay={cheapest['weight_decay']}, train_time_s={cheapest['train_time_s']}.",
    ]
    return to_table("4.18", "Evaluasi kriteria sukses", frame,
                    ["metrics/final_predictions.csv", "best.json", "runs_*.csv", "metrics/inference_benchmark.csv"], notes)


def collect_table_4_19() -> dict:
    """Dimensi perbandingan langsung RM-b dan RM-c."""
    results, values, outcome = compute_test_results(), summarize_efficiency(), evaluate_criteria()
    rows = [
        {"dimensi": "val_f1_macro", "RM-b": float(load_champion_row("rmb")["val_f1_macro"]),
         "RM-c": float(load_champion_row("rmc")["val_f1_macro"])},
        {"dimensi": "test_f1_macro", "RM-b": results["rmb"]["f1_macro"], "RM-c": results["rmc"]["f1_macro"]},
        {"dimensi": "test_f1_judi", "RM-b": results["rmb"]["f1_class1"], "RM-c": results["rmc"]["f1_class1"]},
    ]
    for key in ["trainable_params", "train_time_s", "infer_peak_gpu_mem_mb", "infer_latency_ms", "index_size_mb"]:
        rows.append({"dimensi": key, "RM-b": values["rmb"][key], "RM-c": values["rmc"][key]})
    rows.append({"dimensi": "kriteria_terpenuhi", "RM-b": outcome["rmb"]["kriteria_terpenuhi"],
                 "RM-c": outcome["rmc"]["kriteria_terpenuhi"]})
    frame = pd.DataFrame(rows)
    frame["selisih_rmc_minus_rmb"] = pd.to_numeric(frame["RM-c"], errors="coerce") - pd.to_numeric(frame["RM-b"], errors="coerce")
    notes = ["RM-c memakai head RM-b resmi." if load_best()["rmc"].get("head_is_official_rmb", True)
             else "RM-c memakai head penantang yang BERBEDA dari head RM-b resmi; biaya latihnya milik head itu."]
    return to_table("4.19", "Perbandingan langsung RM-b dan RM-c", frame,
                    ["best.json", "runs_rmb.csv", "runs_rmc.csv", "metrics/final_predictions.csv",
                     "metrics/inference_benchmark.csv", "metrics/index_stats.json"], notes)


def collect_table_l_1() -> dict:
    """Kontribusi RAC per head (lampiran)."""
    summary = summarize_rac_per_head(load_runs("rmc"), load_runs("rmb")).sort_values("rmb_run_id").reset_index(drop=True)
    columns = ["rmb_run_id", "head_arch", "hidden_dim", "lr", "epochs", "trainable_params", "f1_no_rac", "best_alpha",
               "best_k", "f1_best", "gain_best_pp", "f1_shared", "gain_shared_pp", "f1_judi_no_rac",
               "gain_judi_best_pp", "shared_alpha", "shared_k"]
    return to_table("L.1", "Kontribusi RAC per head", summary[columns], ["runs_rmc.csv", "runs_rmb.csv"],
                    ["hidden_dim tidak bermakna untuk head linear. f1_no_rac = baris alpha = 0 (head dimuat dari checkpoint)."])

## 12. Ekspor data tabel

In [ ]:
def save_table_workbook(results: dict, path: Path) -> None:
    """Tulis sheet `keterangan` lalu satu sheet per tabel ke berkas Excel."""
    path.parent.mkdir(parents=True, exist_ok=True)
    generated_at = time.strftime("%Y-%m-%d %H:%M:%S")
    index = [{
        "nomor": number,
        "judul": item["title"] if isinstance(item, dict) else "",
        "status": "tersedia" if isinstance(item, dict) else item,
        "sumber_berkas": "; ".join(item["sources"]) if isinstance(item, dict) else "",
        "catatan": " | ".join(item["notes"]) if isinstance(item, dict) else "",
        "dibangkitkan_pada": generated_at,
    } for number, item in results.items()]
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        pd.DataFrame(index).to_excel(writer, sheet_name="keterangan", index=False)
        for number, item in results.items():
            if isinstance(item, dict):
                item["frame"].to_excel(writer, sheet_name=number, index=False)


table_results = run_builders({
    "4.1": collect_table_4_1, "4.2": collect_table_4_2, "4.3": collect_table_4_3, "4.4": collect_table_4_4,
    "4.5": collect_table_4_5, "4.6": collect_table_4_6, "4.7": collect_table_4_7, "4.8": collect_table_4_8,
    "4.9": collect_table_4_9, "4.10": collect_table_4_10, "4.11": collect_table_4_11, "4.12": collect_table_4_12,
    "4.13": collect_table_4_13, "4.14": collect_table_4_14, "4.15": collect_table_4_15, "4.16": collect_table_4_16,
    "4.17": collect_table_4_17, "4.18": collect_table_4_18, "4.19": collect_table_4_19, "L.1": collect_table_l_1,
})
save_table_workbook(table_results, TABLE_PATH)
print(TABLE_PATH)
for number, item in table_results.items():
    print(f"Tabel {number:5s}: {item if isinstance(item, str) else str(len(item['frame'])) + ' baris'}")

Pratinjau satu tabel; ganti nomornya untuk melihat yang lain. Catatan tiap tabel (misalnya hasil bootstrap Tabel 4.11 dan definisi konfigurasi bersama Tabel 4.12) ada di `notes` dan di sheet `keterangan`.

In [ ]:
NUMBER = "4.17"
item = table_results[NUMBER]
if isinstance(item, str):
    print(item)
else:
    print(item["title"])
    for note in item["notes"]:
        print(" -", note)
    display(item["frame"])

## Ringkasan

Gambar di `outputs/tuning/artifacts/gambar/`, data tabel di `outputs/tuning/artifacts/data_tabel.xlsx`. Keduanya ikut terbungkus arsip di `07_archive.ipynb`.